# DAN Attention Map Visualization

Visualize 4 attention heads của DAN model — xem model tập trung vào vùng nào trên khuôn mặt (mắt, miệng, ...).

In [ ]:
import sys, os
sys.path.insert(0, os.getcwd())

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
import numpy as np
from PIL import Image

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 150

print('PyTorch:', torch.__version__)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

In [ ]:
# ─── DAN Model có return attention map ───
class DANWithAttention(nn.Module):
    def __init__(self, num_class=7, num_head=4):
        super().__init__()
        resnet = models.resnet18(weights=None)
        self.features = nn.Sequential(*list(resnet.children())[:-2])
        self.num_head = num_head
        self.conv_att = nn.Conv2d(512, self.num_head, kernel_size=1)
        self.fc = nn.Linear(512, num_class)
        self.bn = nn.BatchNorm1d(num_class)

    def forward(self, x):
        x = self.features(x)                          # [B, 512, 7, 7]
        att_map = self.conv_att(x)                    # [B, 4, 7, 7] — raw attn scores
        att_map_flat = att_map.view(x.size(0), self.num_head, -1)
        att_weights = F.softmax(att_map_flat, dim=2)  # softmax over spatial
        att_weights = att_weights.view_as(att_map)     # [B, 4, 7, 7]

        # Weighted aggregation
        x_flat = x.view(x.size(0), 1, x.size(1), -1)           # [B, 1, 512, 49]
        att_flat = att_weights.view(x.size(0), self.num_head, 1, -1)  # [B, 4, 1, 49]
        weighted = (x_flat * att_flat).sum(dim=-1)              # [B, 4, 512]
        final = weighted.mean(dim=1)                            # [B, 512]

        out = self.fc(final)
        out = self.bn(out)
        return out, att_weights  # return cả attention maps

In [ ]:
# ─── Load model ───
EMOTIONS = ['Surprise', 'Fear', 'Disgust', 'Happiness', 'Sadness', 'Anger', 'Neutral']
NUM_CLASSES = len(EMOTIONS)

model = DANWithAttention(num_class=NUM_CLASSES, num_head=4)

checkpoint_path = 'outputs/models/best_dan_model.pth'
if os.path.exists(checkpoint_path):
    state = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(state)
    print(f'Loaded: {checkpoint_path}')
else:
    print(f'File not found: {checkpoint_path}')

model.to(device)
model.eval()

In [ ]:
# ─── Image transform ───
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])


def load_image(path):
    img = Image.open(path).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)
    return img, img_tensor


def predict(model, img_tensor):
    with torch.no_grad():
        logits, att_maps = model(img_tensor)
        probs = F.softmax(logits, dim=1).cpu().numpy()[0]
    pred_idx = int(np.argmax(probs))
    conf = probs[pred_idx]
    return pred_idx, conf, probs, att_maps

In [ ]:
# ─── Chọn 2 ảnh mẫu ───
TEST_DIR = 'data/DATASET/test'
sample_images = [
    os.path.join(TEST_DIR, '4', 'test_0003_aligned.jpg'),  # Happiness
    os.path.join(TEST_DIR, '1', 'test_0002_aligned.jpg'),  # Surprise
]

for p in sample_images:
    assert os.path.exists(p), f'Missing: {p}'
print('All sample images found.')

In [ ]:
# ─── Visualize attention map ───
HEAD_LABELS = ['Head 1', 'Head 2', 'Head 3', 'Head 4']
COLORMAP = 'jet'


def visualize_attention(img_pil, att_maps, pred_idx, conf, true_label=None):
    """att_maps: [1, 4, 7, 7] — attention weights cho 4 heads."""
    img_np = np.array(img_pil)
    h, w = img_np.shape[:2]

    n_heads = att_maps.shape[1]
    fig, axes = plt.subplots(2, n_heads + 1, figsize=(18, 8))

    title = f"Pred: {EMOTIONS[pred_idx]} ({conf:.2%})"
    if true_label is not None:
        title += f' | True: {true_label}'
    fig.suptitle(title, fontsize=14, fontweight='bold')

    # Row 0: raw attention maps
    axes[0, 0].imshow(img_np)
    axes[0, 0].set_title('Original', fontsize=10)
    axes[0, 0].axis('off')

    for i in range(n_heads):
        att = att_maps[0, i].cpu().numpy()  # [7, 7]
        att_resized = np.array(Image.fromarray(att).resize((w, h), Image.BILINEAR))

        ax = axes[0, i + 1]
        im = ax.imshow(att_resized, cmap=COLORMAP, interpolation='bilinear')
        ax.set_title(f'{HEAD_LABELS[i]} (raw)', fontsize=10)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)

    # Row 1: overlay trên ảnh gốc
    axes[1, 0].imshow(img_np)
    axes[1, 0].set_title('Original', fontsize=10)
    axes[1, 0].axis('off')

    for i in range(n_heads):
        att = att_maps[0, i].cpu().numpy()  # [7, 7]
        att_resized = np.array(Image.fromarray(att).resize((w, h), Image.BILINEAR))

        ax = axes[1, i + 1]
        ax.imshow(img_np, alpha=0.6)
        im = ax.imshow(att_resized, cmap=COLORMAP, alpha=0.5, interpolation='bilinear')
        ax.set_title(f'{HEAD_LABELS[i]} (overlay)', fontsize=10)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046)

    plt.tight_layout()
    plt.show()


def visualize_combined_attention(img_pil, att_maps, pred_idx, conf, true_label=None):
    """Gộp 4 head thành 1 attention map tổng hợp."""
    img_np = np.array(img_pil)
    h, w = img_np.shape[:2]

    # Tổng hợp: mean của 4 heads
    combined = att_maps[0].mean(dim=0).cpu().numpy()  # [7, 7]
    combined_resized = np.array(Image.fromarray(combined).resize((w, h), Image.BILINEAR))

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    title = f"Pred: {EMOTIONS[pred_idx]} ({conf:.2%})"
    if true_label is not None:
        title += f' | True: {true_label}'
    fig.suptitle(title, fontsize=14, fontweight='bold')

    axes[0].imshow(img_np)
    axes[0].set_title('Original', fontsize=12)
    axes[0].axis('off')

    ax = axes[1]
    im = ax.imshow(combined_resized, cmap=COLORMAP, interpolation='bilinear')
    ax.set_title('Combined Attention (mean)', fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

    ax = axes[2]
    ax.imshow(img_np, alpha=0.6)
    im = ax.imshow(combined_resized, cmap=COLORMAP, alpha=0.5, interpolation='bilinear')
    ax.set_title('Overlay', fontsize=12)
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)

    plt.tight_layout()
    plt.show()

In [ ]:
# ─── Run trên 2 ảnh ───
for img_path in sample_images:
    true_class = os.path.basename(os.path.dirname(img_path))
    true_label = EMOTIONS[int(true_class) - 1]

    img_pil, img_tensor = load_image(img_path)
    pred_idx, conf, probs, att_maps = predict(model, img_tensor)

    print(f"{'='*60}")
    print(f"Image: {os.path.basename(img_path)}")
    print(f"True: {true_label} | Predicted: {EMOTIONS[pred_idx]} ({conf:.2%})")
    print(f"Probabilities:")
    for i, p in enumerate(probs):
        bar = '█' * int(p * 30)
        print(f"  {EMOTIONS[i]:12s}: {p:.2%} {bar}")

    visualize_attention(img_pil, att_maps, pred_idx, conf, true_label)
    visualize_combined_attention(img_pil, att_maps, pred_idx, conf, true_label)

## Nhận xét

- Các attention heads có thể focus vào các vùng khác nhau trên khuôn mặt (mắt, miệng, mũi,...)
- Vùng đỏ/cam = model tập trung nhiều nhất khi đưa ra dự đoán
- Head 1 và Head 2 thường focus vào tổng thể, Head 3-4 vào chi tiết